In [ ]:
#setup
import os

REPO_URL = "https://github.com/meriem200512365/Chat-boot-cegedim.git"
REPO_DIR = "Chat-boot-cegedim"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}

In [ ]:
#chargement du XML
import xml.etree.ElementTree as ET

XML_PATH = "data/menu.xml"

tree = ET.parse(XML_PATH)
root = tree.getroot()

print("Racine du document :", root.tag)
print("Enfants directs de la racine :", [child.tag for child in root])

In [ ]:
#exploration du menu 
menu_module = root.find(".//MenuModule")
main_menu = menu_module.attrib.get("MainMenu")
print("Attribut MainMenu :", main_menu)

In [ ]:
#
menus = root.findall(".//Menu")
print(f"Nombre total de blocs <Menu> : {len(menus)}")

total_items = 0
items_per_menu = {}
for menu in menus:
    name = menu.attrib.get("Name")
    items = menu.findall("MenuItem")
    items_per_menu[name] = len(items)
    total_items += len(items)

print(f"Nombre total de <MenuItem> (tous menus confondus) : {total_items}")
print(f"Moyenne d'items par menu : {total_items/len(menus):.1f}")

In [ ]:
#
import pandas as pd
import matplotlib.pyplot as plt

df_menus = pd.Series(items_per_menu).sort_values(ascending=False)
print(df_menus.describe())

plt.figure(figsize=(10, 4))
df_menus.head(20).plot(kind="bar")
plt.title("Top 20 des menus avec le plus d'items")
plt.ylabel("Nombre de MenuItem")
plt.xlabel("Nom du menu")
plt.xticks(rotation=75, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
noeuds = 0
feuilles = 0
sans_label = []
sans_id = []

for menu in menus:
    for item in menu.findall("MenuItem"):
        name = item.attrib.get("Name")
        label = item.attrib.get("Label")
        submenu = item.attrib.get("SubMenuName")

        if submenu:
            noeuds += 1
        else:
            feuilles += 1

        if not label:
            sans_label.append((menu.attrib.get("Name"), name))
        if not name:
            sans_id.append((menu.attrib.get("Name"), label))

print(f"Items 'noeud' (avec SubMenuName)  : {noeuds}")
print(f"Items 'feuille' (destination finale) : {feuilles}")
print(f"\nItems sans Label : {len(sans_label)}")
print(f"Items sans Name (id) : {len(sans_id)}")

if sans_label:
    print("\nExemples d'items sans label (menu, name) :")
    for m, n in sans_label[:10]:
        print("  -", m, n)


In [ ]:
from collections import deque, defaultdict

# graphe : menu -> liste des sous-menus qu'il reference
graph = defaultdict(list)
menu_names = set()
for menu in menus:
    name = menu.attrib.get("Name")
    menu_names.add(name)
    for item in menu.findall("MenuItem"):
        sub = item.attrib.get("SubMenuName")
        if sub:
            graph[name].append(sub)

# BFS depuis MainMenu
depth = {main_menu: 0}
queue = deque([main_menu])
while queue:
    current = queue.popleft()
    for nxt in graph[current]:
        if nxt not in depth:
            depth[nxt] = depth[current] + 1
            queue.append(nxt)

depth_series = pd.Series(depth)
print("Profondeur max atteinte :", depth_series.max())
print("\nRepartition des menus par profondeur :")
print(depth_series.value_counts().sort_index())

plt.figure(figsize=(6, 4))
depth_series.value_counts().sort_index().plot(kind="bar")
plt.title("Nombre de menus par profondeur depuis MainMenu")
plt.xlabel("Profondeur")
plt.ylabel("Nombre de menus")
plt.tight_layout()
plt.show()


In [ ]:
#orphelins
orphelins = menu_names - set(depth.keys())
print(f"Menus definis : {len(menu_names)}")
print(f"Menus atteints depuis MainMenu : {len(depth)}")
print(f"Menus orphelins : {len(orphelins)}")
if orphelins:
    print(sorted(orphelins))
